# ProtectRAG Eval Dashboard

Visual analysis of golden-set evaluation results.

Run this notebook after `pip install -e '.[dev]'` in the ProtectRAG repo.

In [ ]:
import sys
sys.path.insert(0, '../src')

from protectrag.datasets import load_golden_v1
from protectrag.evals import run_eval_dataset, EvalReport
from protectrag.scanner import scan_document_for_injection

cases = load_golden_v1()
print(f'Loaded {len(cases)} cases')

In [ ]:
report = run_eval_dataset(
    cases,
    classify=lambda t, cid: scan_document_for_injection(t, document_id=cid),
    run_id='notebook-run',
    project='protectrag-golden',
)

summary = report.to_dict()
print('=== Eval Summary ===')
for k, v in summary.items():
    if k != 'case_results':
        print(f'  {k}: {v}')

In [ ]:
# Confusion matrix
print(f'\nConfusion matrix (labeled cases):')
print(f'  TP={report.true_positives}  FP={report.false_positives}')
print(f'  FN={report.false_negatives}  TN={report.true_negatives}')
print(f'\nPrecision: {report.precision}')
print(f'Recall:    {report.recall}')
print(f'Accuracy:  {report.accuracy}')

In [ ]:
# Per-case breakdown
false_negatives = [cr for cr in report.case_results if cr.ground_truth.value == 'injection' and not cr.predicted_alert]
false_positives = [cr for cr in report.case_results if cr.ground_truth.value == 'clean' and cr.predicted_alert]

if false_negatives:
    print('\n=== FALSE NEGATIVES (missed injections) ===')
    for cr in false_negatives:
        c = next(c for c in cases if c.id == cr.case_id)
        print(f'  [{cr.case_id}] severity={cr.severity.name} text={c.text[:80]}...')

if false_positives:
    print('\n=== FALSE POSITIVES (benign flagged as risky) ===')
    for cr in false_positives:
        c = next(c for c in cases if c.id == cr.case_id)
        print(f'  [{cr.case_id}] severity={cr.severity.name} text={c.text[:80]}...')

if not false_negatives and not false_positives:
    print('\nPerfect score on this dataset.')

In [ ]:
# Severity distribution
from collections import Counter

sev_counts = Counter(cr.severity.name for cr in report.case_results)
print('\n=== Severity distribution ===')
for sev in ['NONE', 'LOW', 'MEDIUM', 'HIGH']:
    count = sev_counts.get(sev, 0)
    bar = '█' * count
    print(f'  {sev:6s} | {bar} ({count})')